<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-28</br>
</div>

</br>

In [ ]:
# TODO 0: 디바이스를 확인하고, FP16 모델과 INT4 양자화 모델을 각각 로드해봅시다.


# 디바이스 확인


# 추론 헬퍼 함수 (chat template 적용)

# --- FP16 모델 로드 ---

# FP16 응답을 미리 생성하여 저장 (GPU 메모리 절약)



</br>

# 학습 내용
>이번 장에서는 <strong>TTP 전략(Test-Time Prompting Strategy)</strong>에 대해 학습합니다.</br></br>
>추론 시점에서 프롬프트를 최적화하여 모델 성능을 향상시키는 전략을 학습하고, 환경 변화 입력에 TTP를 적용하여 품질 개선 효과를 정량적으로 비교해봅시다.

</br>

# TTP (Test-Time Prompting)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">추론(테스트) 시점에서 프롬프트를 정교하게 설계</mark>하여 모델의 응답 품질을 높이는 전략입니다.</br></br>
> 모델 재학습 없이 프롬프트만으로 성능을 개선합니다.

> 추론 시점에서 추가 처리가 필요한 이유는 세 가지입니다.</br>
> 첫째, 모델이 학습 데이터에 없는 특수한 도메인이나 형식을 요구받을 때 프롬프트로 맥락을 보완할 수 있습니다.</br>
> 둘째, 동일한 질문도 프롬프트 구성 방식에 따라 응답 품질이 크게 달라지므로, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">규칙 명시, 역할 부여, 출력 형식 지정</mark>으로 원하는 답변 패턴을 유도합니다.</br>
> 셋째, 같은 질문을 다양한 프롬프트 방식으로 여러 번 질의하고 결과를 종합하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">더 안정적이고 정확한 답변</mark>을 얻는 앙상블 효과를 기대할 수 있습니다.</br></br>
> TTP는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">재학습 비용 없이 모델 성능을 즉시 개선</mark>할 수 있는 가장 빠르고 저렴한 방법입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">접근 방식</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">모델 학습</td><td>가중치 변경 (비용 높음)</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">TTP</mark></td><td>프롬프트 변경 (비용 낮음)</td></tr>
    <tr><td style="text-align:center">RAG</td><td>외부 지식 추가</td></tr>
  </tbody>
</table>

</br>

## 규칙 명시 (Rule Specification)
> 모델에게 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">따라야 할 규칙을 명확하게 지시</mark>합니다.

In [ ]:
# TODO 1: FP16 모델로 정상 질문의 응답을 생성하고 캐시에 저장해봅시다.



</br>

## Few-shot TTP
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">도메인에 특화된 예시</mark>를 프롬프트에 포함하여 출력 패턴을 유도합니다.

In [ ]:
# TODO 2: 오타/노이즈 입력의 FP16 응답을 캐시하고, FP16 해제 후 INT4를 로드해봅시다.


# FP16 응답 캐시

# FP16 모델 해제 → INT4 로드 (GPU 메모리 절약)

quantization_config = BitsAndBytesConfig(
    model_name, quantization_config=quantization_config, device_map="auto"

# FP16 vs INT4 비교 출력



</br>

## 도메인별 TTP 전략

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">도메인</th>
      <th>전략</th>
      <th>핵심</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">고객 서비스</td><td>규칙 + Few-shot</td><td>정확성, 정중함</td></tr>
    <tr><td style="text-align:center">코드 생성</td><td>명세 + 제약 조건</td><td>실행 가능한 코드</td></tr>
    <tr><td style="text-align:center">의료/법률</td><td>면책 조항 + 출처 요구</td><td>안전성, 신뢰성</td></tr>
    <tr><td style="text-align:center">데이터 분석</td><td>출력 형식 지정</td><td>구조화된 결과</td></tr>
  </tbody>
</table>

</br>

## 프롬프트 엔지니어링 패턴

In [ ]:
# TODO 3: 프롬프트 엔지니어링 4가지 패턴(역할 부여, 출력 형식, CoT, 자기 검증)을 조합하여 INT4 모델의 응답을 개선해봅시다.

combined_prompt = """당신은 한국 교통 전문가입니다.

규칙:
1. 한국어로 간결하게 답변하세요.
2. 소요 시간과 비용을 포함하세요.
3. 단계별로 설명하세요.

질문: 서울에서 부산까지 KTX로 얼마나 걸리나요?"""

int4_basic = generate(model_int4, "서울에서 부산까지 KTX로 얼마나 걸리나요?")
int4_enhanced = generate(model_int4, combined_prompt)

print("[INT4 - 기본 프롬프트]")
print(f"  {int4_basic[:150]}")
print()
print("[INT4 - 엔지니어링 프롬프트]")
print(f"  {int4_enhanced[:150]}")
print()
print("→ 프롬프트 설계만으로 INT4 응답 품질이 개선됩니다. 이것이 TTP의 핵심입니다.")

💡TTP 효과 극대화
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">규칙 명시 + Few-shot + 출력 형식 지정</mark>을 조합하면 가장 좋은 결과를 얻습니다.

<div style="text-align:center">

</div>

</br>

# TTP 적용 전후 품질 비교 (정량 평가)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정상/오타/모호 입력 각각에 TTP를 적용한 전후 결과</mark>를 나란히 놓고 정량적으로 비교합니다.</br></br>
> Ch.5-2_002에서 관찰한 환경 변화 입력(오타, 노이즈, 모호함)의 품질 저하를 TTP 전략으로 얼마나 회복할 수 있는지 측정합니다.</br></br>
> TTP-A(규칙 명시)와 TTP-B(Few-shot 예시)를 적용하고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력 유형별 품질 점수를 비교</mark>하여 TTP의 실질적 효과를 체감합니다.

## TTP 템플릿 정의

In [ ]:
# TODO 4: TTP-A(규칙 명시) TTP 템플릿을 정의해봅시다.

# TTP-A: 출력 형식/제약 강화
TTP_A_TEMPLATE = """다음 질문에 답해주세요.

**규칙:**
1. 오타나 불명확한 표현이 있어도 의도를 파악하세요.
2. 교통 관련 질문은 대략적인 소요 시간(시간 단위)으로 답하세요.
3. 맥락이 부족하면 일반적인 상황을 가정하여 답하세요.
4. 간결하게 핵심만 답하세요.

**질문:** {question}

**답변:**"""

print("TTP-A 템플릿 정의 완료: 출력 형식/제약 강화 (규칙 4개 명시)")

In [ ]:
# TODO 5: TTP-B(Few-shot 예시) TTP 템플릿을 정의해봅시다.

# TTP-B: Few-shot 예시 포함
TTP_B_TEMPLATE = """다음은 교통 관련 질문과 답변의 예시입니다.

**예시:**
Q: 서울역에서 대전역까지 KTX로 얼마나 걸려요?
A: 서울역에서 대전역까지 KTX로 약 50분 소요됩니다.

Q: 인천공항에서 서울역까지 어떻게 가나요?
A: 인천공항에서 서울역까지 공항철도 직통열차로 약 43분 소요됩니다.

**질문:** {question}

**답변:**"""

print("TTP-B 템플릿 정의 완료: Few-shot 예시 포함 (교통 Q&A 2개)")

</br>

## 환경 변화 입력에 TTP 적용

In [ ]:
# TODO 6: 환경 변화 입력 샘플 5종을 딕셔너리 리스트로 정의해봅시다.

test_samples = [
    {"type": "정상",   "prompt": "서울에서 부산까지 KTX로 얼마나 걸리나요?"},
    {"type": "오타",   "prompt": "서울에서 부싼까지 KTX로 얼마나 걸리나여?"},
    {"type": "노이즈", "prompt": "서울에서 부산까지... 음... KTX로 얼마나 걸리나요? 대략?"},
    {"type": "모호함", "prompt": "그거 얼마나 걸려?"},
    {"type": "조건변화", "prompt": "밤 11시에 출발하면 다음날 아침에 도착할 수 있나요?"},
]

print(f"환경 변화 입력 샘플: {len(test_samples)}개")
for s in test_samples:
    print(f"  [{s['type']}] {s['prompt']}")

In [ ]:
# TODO 7: TTP 템플릿에 질문을 삽입하는 apply_ttp 헬퍼 함수를 정의해봅시다.

def apply_ttp(template, question):
    """TTP 템플릿에 질문을 삽입합니다."""
    return template.format(question=question)

print("apply_ttp 함수 정의 완료")

In [ ]:
# TODO 8: 환경 변화 입력에 대해 FP16(캐시) / INT4 / INT4+TTP-A / INT4+TTP-B 4가지를 비교해봅시다.






</br>

</br>

</br>

# 도메인별 TTP 설계 (Domain-Specific TTP)
> 의료 도메인에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">TTP 전략을 적용</mark>하여 도메인 특화 프롬프트의 효과를 체감합니다.

In [ ]:
# TODO 9: 의료 도메인 TTP-A(규칙 명시) 템플릿을 정의해봅시다.

MEDICAL_TTP_A = """다음 의료 관련 질문에 답해주세요.

**규칙:**
1. 오타나 구어체 표현이 있어도 증상을 정확히 파악하세요.
2. 답변 형식: "증상 → 가능한 원인 → 권장 조치" 순서로 답하세요.
3. 심각한 증상이 의심되면 반드시 "병원 방문을 권장합니다"를 포함하세요.
4. 의학적으로 확인되지 않은 민간요법은 권장하지 마세요.

**질문:** {question}

**답변:**"""

print("의료 도메인 TTP-A 템플릿 정의 완료")

In [ ]:
# TODO 10: 의료 도메인 TTP-B(Few-shot 예시) 템플릿을 정의해봅시다.

MEDICAL_TTP_B = """다음은 의료 관련 질문과 답변의 예시입니다.

**예시:**
Q: 며칠째 두통이 심해요.
A: 증상: 지속적 두통 → 가능한 원인: 긴장성 두통, 편두통, 수면 부족 → 권장 조치: 충분한 휴식과 수분 섭취를 하시고, 3일 이상 지속되면 병원 방문을 권장합니다.

Q: 배가 아프고 설사를 해요.
A: 증상: 복통 + 설사 → 가능한 원인: 급성 장염, 식중독 → 권장 조치: 수분 보충(이온 음료)을 하시고, 고열이나 혈변이 있으면 즉시 병원 방문을 권장합니다.

**질문:** {question}

**답변:**"""

print("의료 도메인 TTP-B 템플릿 정의 완료")

In [ ]:
# TODO 11: 의료 도메인 환경 변화 입력에 TTP를 적용하고 결과를 비교해봅시다.





💡도메인별 TTP 설계 원칙
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">도메인마다 규칙과 예시가 달라야</mark> 합니다.</br>
> 의료: 안전성(병원 방문 권고) 우선, 고객 서비스: 정확성 + 정중함 우선, 코드 생성: 실행 가능성 우선.</br>
> TTP 템플릿은 해당 도메인 전문가와 함께 설계하는 것이 가장 효과적입니다.

💡TTP 전략 선택 가이드
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">TTP-A(규칙 명시)</mark>는 구현이 간단하고 토큰 비용이 낮아 빠르게 적용할 수 있습니다.</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">TTP-B(Few-shot)</mark>는 더 높은 품질 개선을 보이지만, 예시가 추가되어 토큰 비용이 증가합니다.</br>
> 실무에서는 간단한 TTP-A부터 시작하고, 품질이 부족하면 TTP-B로 확장하는 것을 권장합니다.

💡양자화 모델에서의 TTP
> INT4 양자화 모델은 정밀도 손실로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">노이즈 입력에 취약</mark>합니다.</br>
> 더 명확하고 구조화된 프롬프트가 필요합니다.

</br>

## 운영 가이드: 양자화 + TTP 조합 전략

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">시나리오</th>
      <th style="text-align:center">양자화</th>
      <th style="text-align:center">TTP</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">프로토타이핑</td><td style="text-align:center">INT4</td><td style="text-align:center">TTP-A</td><td>빠른 실험, 최소 메모리</td></tr>
    <tr><td style="text-align:center">프로덕션 (품질 우선)</td><td style="text-align:center">FP16</td><td style="text-align:center">TTP-B</td><td>최고 품질, 메모리 여유 필요</td></tr>
    <tr><td style="text-align:center">프로덕션 (균형)</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">INT4 + QLoRA</mark></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">TTP-B</mark></td><td>메모리 절약 + 도메인 특화 + 품질 보완</td></tr>
    <tr><td style="text-align:center">엣지 디바이스</td><td style="text-align:center">INT4</td><td style="text-align:center">TTP-A</td><td>최소 리소스, 규칙 기반 보완</td></tr>
  </tbody>
</table>